In [5]:
using LinearAlgebra, Polynomials, Plots
using Revise, DelimitedFiles, BenchmarkTools
using CloudAtlas, BifurcationKit, ChannelflowWrapper
using Dates
using Random
using Base.Threads

"""
    myreaddlm(filename, cc='%')

Read matrix or vector from a file, dropping comments marked with cc.
"""
function myreaddlm(filename; cc='%')
    X = readdlm(filename, comments=true, comment_char=cc)
    if size(X,2) == 1
        X = X[:,1]
    end
    X
end

macro suppress(ex)
    quote
        # Generate a unique name for the old stdout to avoid variable collision
        local old_stdout = stdout
        redirect_stdout(devnull)
        try
            # We use esc(ex) to run the expression in the caller's scope
            $(esc(ex))
        finally
            redirect_stdout(old_stdout)
        end
    end
end

sx, sy, sz, tx, tz = halfbox_symmetries()

pwd()

"/home/ebenq/Dev/julia/CloudAtlas.jl/notebooks/tw_fuzzing-updated"

In [6]:
# Parameters
hookparams = SearchParams(ftol=1e-08, xtol=1e-12, Nnewton=30,Nhook=8,δ=0.01, verbosity=0)
Re = 300.0
cx0 = 0.000 # values from paper
cz0 = 0.009

α, γ = 2π/4.0, 2π/6.0               # Fourier wavenumbers α, γ = 2π/Lx, 2π/Lz
normalize = true                    # Normalize the basis set or not?
H = [(sx*sy)*(tx*tz)]               # Generators of the symmetric subspace of TW1

# The DNS file to project from (ensure this path is correct)
dns_file = "TW1-2pi1piRe200-40x49x40.nc" 

# List of resolutions to test: [(J, K, L), ...]
# discretizations = [(1, 1, 1), (1, 1, 2), (1, 1, 3), (1, 2, 3), (1, 3, 5), (2, 4, 7), (3, 5, 9)]
# discretizations = [(1, 1, 3), (1, 2, 3), (1, 3, 5), (2, 4, 7)]
# discretizations = [(1, 1, 3), (1, 2, 3)]
discretizations = [(1, 1, 3)]

1-element Vector{Tuple{Int64, Int64, Int64}}:
 (1, 1, 3)

In [7]:
struct SolutionFingerprint
    cx::Float64
    cz::Float64
    nm::Float64
end

# Helper to check if a solution is effectively new
function is_distinct(new_fp::SolutionFingerprint, archive::Vector{SolutionFingerprint}; tol=1e-4)
    for fp in archive
        # If wave speeds AND norm are identical (within tolerance), it's a duplicate
        if isapprox(new_fp.cx, fp.cx, atol=tol) && 
           isapprox(new_fp.cz, fp.cz, atol=tol) && 
           isapprox(new_fp.nm, fp.nm, atol=tol)
            return false # It's a duplicate
        end
    end
    return true
end

is_distinct (generic function with 1 method)

In [8]:
function fuzz_symmetry_space(discretizations, H, Re; 
                             attempts_per_level=25, 
                             base_dir="fuzz_results",
                             symm_file = "./sxytxz.asc",
                             reference_path = "./TW1-2pi1piRe200-40x49x40.nc",
                             norm_threshold=1e-2,
                             xnorm = 0.40,
                             α=α, γ=γ, hookparams=hookparams, T=10.0)
    
    # 1. Setup Directory
    mkpath(base_dir)
    
    # --- Thread Safety Tools ---
    io_lock = ReentrantLock()        # For printing
    data_lock = ReentrantLock()      # NEW: For accessing the solution archive
    total_found = Atomic{Int}(0)
    
    # NEW: Archive to store fingerprints of found solutions
    # We store tuples of (cx, cz, norm) to quickly identify duplicates
    solution_archive = Vector{SolutionFingerprint}()

    reference_field_converted = "reference_field_$(α)_$(γ).nc"
    changegrid(reference_path, reference_field_converted; al=α, ga=γ)
    
    println("Starting Smart Fuzz Search in $base_dir with $(nthreads()) threads")

    for (J, K, L) in discretizations
        lock(io_lock) do 
            println("\n" * "="^60)
            println("  Discretization: J=$J, K=$K, L=$L")
            println("="^60)
        end

        # Pre-calculate model for this level
        model = TWModel(α, γ, J, K, L, H; normalize=false)
        m = length(model)

        @threads for i in 1:attempts_per_level
            
            # ... [Random Guess Generation code remains the same] ...
            x_guess = randn(m)
            x_guess = xnorm/norm(x_guess) * x_guess
            cx_guess = randn() * 0.1
            cz_guess = randn() * 0.1
            ξ_guess = [x_guess; cx_guess; cz_guess]

            # Define closures for solver
            f(ξ) = model.g(ξ, Re)
            Df(ξ) = model.Dg(ξ, Re)
            
            # Try Low-Dimensional Solve
            ξ_star, converged = hookstepsolve(f, Df, ξ_guess, hookparams)

            # Check convergence basics
            solution_norm = norm(ξ_star[1:m])
            
            if converged && solution_norm > norm_threshold && ξ_star[end - 1] > 1e-7
                
                x_found, cx_found, cz_found = extract_components(ξ_star, model)

                # Create a fingerprint for this solution
                new_fp = SolutionFingerprint(cx_found, cz_found, solution_norm)
                
                is_new = false
                lock(data_lock) do
                    if is_distinct(new_fp, solution_archive)
                        push!(solution_archive, new_fp)
                        is_new = true
                    end
                end
                
                if !is_new
                    # Skip expensive findsoln if we've seen this wave before
                    # Optional: Print a "skip" message if you want to track efficiency
                    # lock(io_lock) do println("  [Thread $(threadid())] Skipped duplicate (cx=$cx_found)") end
                    continue 
                end

                # --- Proceed to Save and Refine (Only for unique solutions) ---
                atomic_add!(total_found, 1)
                
                lock(io_lock) do
                    println("  [Thread $(threadid())] Hit! Unique Solution found (cx=$(round(cx_found, digits=5)))")
                end
                
                # ... [File saving and findsoln call code remains the same] ...
                timestamp = Dates.format(now(), "MM-DD-HHMMSS")
                sol_dir = joinpath(base_dir, "sol_$(J)_$(K)_$(L)_id$(i)_$(timestamp)")
                mkpath(sol_dir)
                guess_path = joinpath(sol_dir, "u_guess.nc")
                sigma_file = joinpath(sol_dir, "sigma.asc")
                
                lock(io_lock) do
                    coeff2field(ξ_star[1:m], model.ijkl, reference_field_converted, guess_path)
                    save_sigma(model, ξ_star[end - 1], ξ_star[end], T, sigma_file)
                end
                
                try
                    findsoln(guess_path;
                        R = Re, eqb = true, xrel = model.keep_cx, zrel = model.keep_cz,
                        symms = abspath(symm_file), sigma = sigma_file, od = sol_dir, T = T
                    )
                catch e
                    lock(io_lock) do
                        println("  [Thread $(threadid())] findsoln failed: $e")
                    end
                end
            end
        end
    end
end

fuzz_symmetry_space (generic function with 1 method)

In [ ]:
fuzz_symmetry_space(discretizations, H, Re; attempts_per_level=50, α=α, γ=γ, hookparams=hookparams)

Leaving rescale
L2Norm(u0)  == 0.2792590266581449
L2Norm(u1)  == 0.2792590266581449
bcNorm(u0)  == 8.737886593015271e-17
bcNorm(u1)  == 1.001456466862179e-16
divNorm(u0) == 4.746636220033689e-16
divNorm(u1) == 6.392261886532839e-16
L2Norm(u2)  == 0.2792590266581449
divNorm(u2) == 6.271116005970482e-18
bcNorm(u2)  == 4.580081077120656e-17
Starting Smart Fuzz Search in fuzz_results with 1 threads

  Discretization: J=1, K=1, L=3
J,K,L,m == 1,1,3,33
(2J+1)(2K+1)(2L+1) + 1 == 64
Making matrices B, A1, A2, Cx, Cz...
Phase constraints: keep_cx = false, keep_cz = true
Making quadratic operator N...
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 
  [Thread 1] Hit! Unique Solution found (cx=0.04201)
alpha, gamma == 1.570796326794897, 1.047197551196598
Nx, Ny, Nz == 40, 49, 40
Reading ijkl indices of basis set from file
reading N == 33 ijkl indices
ijkl[0] == 1 0 0 1
L == max l == 3
Constructing Legendre polynomials
Assigning Polynomial, size = 1 
Assig

L2Norm(uout)   == 0.09958671297066411
L2Dist(u,uout) == 0.3149667313113736
L2Dist(u,uout)/L2Norm(u) == 1.127865892395809
nu==0.003333333333333334, Vsuck==0, rotation==0, theta==0, dPdx==0, dPdz==0, Ubulk==0, Wbulk==0, uwall==1, uupper==1, ulower==-1, wupper==0, wlower==-0, t0==0, dT==1, dt==0.03125, variabledt==1, dtmin==0.001, dtmax==0.2, CFLmin==0.4, CFLmax==0.6, LaminarBase, PressureGradient, SBDF3, SMRK2, Rotational, DealiasXZ, zero_bodyforce, TauCorrection, Silent
57503 unknowns
Computing G(x)
f^T: .....10
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0630915
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.00968337
   previous gx == 0.00968337
   current  gx == 0.00968337
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 4.68839e-05
   previous rx == 4.68839e-05
   current  rx == 4.68839e-05
         delta == 0.01
Newt,GMRES == 0,0, f^

f^T: .......10 res == 0.122801
Newt,GMRES == 19,13, 10f^T: 
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0630914
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.00949883
   previous gx == 0.00949883
   current  gx == 0.00949883
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 4.51139e-05
   previous rx == 4.51139e-05
   current  rx == 4.51139e-05
         delta == 0.01
Newt,GMRES == 0,0, f^T: .......10. res == 0.0838514
Newt,GMRES == 19,14, f^T: .10 res == 0.791551
Newt,GMRES == 0,1, f^T: ......10. res == 0.043355
Newt,GMRES == 19,15, f^T: .10 res == 0.439691
Newt,GMRES == 0,2, f^T: ......10 res == 0.0220279
Newt,GMRES == 19,16, .f^T: .10 res == 0.328164
Newt,GMRES == 0,3, f^T: .......10 res == 0.264688
Newt,GMRES == 0,4, 10f^T: . res == 0.0115524
Newt,GMRES == 19,17, f^T: .......10 res == 0.227702
Newt,GMRES == 0,5, f^T: .10 res == 0.006

f^T: ....10. res == 0.00261767
Newt,GMRES == 0,19, f^T: ...10.
Newton iteration number 0
Current state of Newton iteration:
   fcount_newton   == 1
   fcount_optimiza == 0
   L2Norm(x)       == 0.0630914
   L2Norm(dxN)     == 0
   L2Norm(dxOpt)   == 0
   L2Dist(x,x0)    == 0
gx == L2Norm(G(x)) : 
   initial  gx == 0.00970493
   previous gx == 0.00970493
   current  gx == 0.00970493
rx == 1/2 L2Norm2(G(x)) : 
   initial  rx == 4.70928e-05
   previous rx == 4.70928e-05
   current  rx == 4.70928e-05
         delta == 0.01
Newt,GMRES == 0,0, f^T: ...10. res == 0.00212753
Newt,GMRES == 0,20, f^T: ....10 res == 0.818962
Newt,GMRES == 0,1, f^T: ...10. res == 0.00172156
Newt,GMRES == 0,21, f^T: ..10 res == 0.43694
Newt,GMRES == 0,2, f^T: ......10. res == 0.00147344
Newt,GMRES == 0,22, f^T: 10. res == 0.317536
Newt,GMRES == 0,3, f^T: .......1010 res == 0.287491
Newt,GMRES == 0,4, f^T:  res == 0.00129014
Newt,GMRES == 0,23, f^T: .......10 res == 0.262439
Newt,GMRES == 0,5, f^T: ...10. res == 0.0

Excessive output truncated after 524320 bytes.


Determining what to do with current Newton/hookstep and trust region
rx       ==    3.3367e-11 residual at current position
rH       ==     3.281e-11 residual of newton/hookstep
Delta_rH ==   -5.5697e-13 actual improvement in residual from newton/hookstep

             -5.59712e-16 lower bound for acceptable improvement
             -5.57068e-14 upper bound for ok improvement.
             -4.17801e-13 upper bound for good improvement.
             -5.01361e-13 upper bound for accurate prediction.
Delta_rH ==   -5.5697e-13 ------> Accurate <------
             -6.12775e-13 lower bound for accurate prediction.
             -5.59712e-13 local linear model of improvement
lambda       == 102.069 is the reduction/increase factor for delta suggested by quadratic model
lambda*delta == 0.00896519 is the delta suggested by quadratic model
Improvement is Accurate
  old delta == 8.78346e-05


LoadError: InterruptException: